In [1]:
# Step 1: Libraries install + import
!pip install requests beautifulsoup4 -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time

print("Libraries ready!")


Libraries ready!


In [7]:
# Step 2: Books scrape cheyyadam - 3 categories
BASE_URL = "http://books.toscrape.com/catalogue/category/books/"

# 3 categories select chesam
categories = {
    "mystery": "mystery_3",
    "science": "science_8",
    "travel": "travel_2"
}

def get_books_from_category(category_name, category_slug):
    books = []
    page = 1

    while True:
        if page == 1:
            url = f"{BASE_URL}{category_slug}/index.html"
        else:
            url = f"{BASE_URL}{category_slug}/page-{page}.html"

        response = requests.get(url)

        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.text, "html.parser")
        book_list = soup.find_all("article", class_="product_pod")

        if not book_list:
            break

        for book in book_list:
            title = book.h3.a["title"]
            price = book.find("p", class_="price_color").text.strip()
            rating = book.p["class"][1]  # "Three", "Five" etc
            availability = book.find("p", class_="instock availability").text.strip()

            books.append({
                "title": title,
                "price_raw": price,
                "star_rating": rating,
                "availability_raw": availability,
                "category": category_name
            })

        page += 1
        time.sleep(0.5)

    return books
all_books = []
for cat_name, cat_slug in categories.items():
    print(f"Scraping {cat_name}...")
    books = get_books_from_category(cat_name, cat_slug)
    all_books.extend(books)
    print(f"  {len(books)} books found")

print(f"\nTotal books scraped: {len(all_books)}")
df_raw = pd.DataFrame(all_books)
df_raw.head()

Scraping mystery...
  32 books found
Scraping science...
  0 books found
Scraping travel...
  11 books found

Total books scraped: 43


,title,price_raw,star_rating,availability_raw,category
0,Sharp Objects,Â£47.82,Four,In stock,mystery
1,"In a Dark, Dark Wood",Â£19.63,One,In stock,mystery
2,The Past Never Ends,Â£56.50,Four,In stock,mystery
3,A Murder in Time,Â£16.64,One,In stock,mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Â£44.10,Four,In stock,mystery


In [10]:
all_books_80 = []

for category in df_raw["category"].unique():
    category_books = df_raw[df_raw["category"] == category]
    twenty_books = category_books.head(20)
    all_books_80.append(twenty_books)

df_raw = pd.concat(all_books_80).reset_index(drop=True)

print(df_raw["category"].value_counts())
print(f"\nTotal books: {len(df_raw)}")

category
historical-fiction    20
mystery               20
romance               20
travel                11
Name: count, dtype: int64

Total books: 71


In [11]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

df_clean = df_raw.copy()

df_clean["price_gbp"] = df_clean["price_raw"].str.replace(r"[^\d.]", "", regex=True).astype(float)

df_clean["rating"] = df_clean["star_rating"].map(rating_map)

df_clean["in_stock"] = df_clean["availability_raw"].str.contains("In stock", case=False)

GBP_TO_INR = 105.50
df_clean["price_inr"] = (df_clean["price_gbp"] * GBP_TO_INR).round(2)

df_clean = df_clean.drop(columns=["price_raw", "availability_raw", "star_rating"])

print("Missing values:")
print(df_clean.isnull().sum())
print(f"\nShape: {df_clean.shape}")
print(f"\nData types:")
print(df_clean.dtypes)
df_clean.head()

Missing values:
title        0
category     0
price_gbp    0
rating       0
in_stock     0
price_inr    0
dtype: int64

Shape: (71, 6)

Data types:
title         object
category      object
price_gbp    float64
rating         int64
in_stock        bool
price_inr    float64
dtype: object


,title,category,price_gbp,rating,in_stock,price_inr
0,A Paris Apartment,historical-fiction,39.01,4,True,4115.55
1,The Red Tent,historical-fiction,35.66,5,True,3762.13
2,Tipping the Velvet,historical-fiction,53.74,1,True,5669.57
3,Lost Among the Living,historical-fiction,27.70,4,True,2922.35
4,The Invention of Wings,historical-fiction,37.34,1,True,3939.37


In [14]:
import sqlite3

conn = sqlite3.connect("zepto_books.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

for i, cat in enumerate(df_clean["category"].unique(), start=1):
    cursor.execute("INSERT INTO categories VALUES (?, ?)", (i, cat))

cat_map = {}
for row in cursor.execute("SELECT * FROM categories"):
    cat_map[row[1]] = row[0]

for i, row in df_clean.iterrows():
    cursor.execute("INSERT INTO books VALUES (?, ?, ?, ?, ?, ?, ?)", (
        i + 1,
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),
        cat_map[row["category"]]
    ))

conn.commit()
print("Database ready!")
print(f"Categories: {cursor.execute('SELECT COUNT(*) FROM categories').fetchone()[0]}")
print(f"Books: {cursor.execute('SELECT COUNT(*) FROM books').fetchone()[0]}")

Database ready!
Categories: 4
Books: 71


In [15]:
print("Query 1: In stock books")
q1 = pd.read_sql("""
    SELECT title, price_gbp, rating
    FROM books
    WHERE in_stock = 1
    LIMIT 5
""", conn)
print(q1)

print("\nQuery 2: Books by price low to high")
q2 = pd.read_sql("""
    SELECT title, price_gbp
    FROM books
    ORDER BY price_gbp ASC
    LIMIT 5
""", conn)
print(q2)

print("\nQuery 3: Distinct categories")
q3 = pd.read_sql("""
    SELECT DISTINCT category_name
    FROM categories
""", conn)
print(q3)

print("\nQuery 4: Books between 20 and 40 GBP")
q4 = pd.read_sql("""
    SELECT title, price_gbp, rating
    FROM books
    WHERE price_gbp BETWEEN 20 AND 40
    ORDER BY price_gbp
    LIMIT 5
""", conn)
print(q4)

print("\nQuery 5: Books with category name (JOIN)")
q5 = pd.read_sql("""
    SELECT b.title, b.price_gbp, b.rating, c.category_name
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
    ORDER BY b.rating DESC
    LIMIT 10
""", conn)
print(q5)

Query 1: In stock books
                    title  price_gbp  rating
0       A Paris Apartment      39.01       4
1            The Red Tent      35.66       5
2      Tipping the Velvet      53.74       1
3   Lost Among the Living      27.70       4
4  The Invention of Wings      37.34       1

Query 2: Books by price low to high
                                        title  price_gbp
0                  Hide Away (Eve Duncan #20)      11.84
1  The Purest Hook (Second Circle Tattoos #3)      12.25
2         Changing the Game (Play by Play #2)      13.38
3                           Playing with Fire      13.71
4   The Girl In The Ice (DCI Erika Foster #1)      15.85

Query 3: Distinct categories
        category_name
0  historical-fiction
1             mystery
2             romance
3              travel

Query 4: Books between 20 and 40 GBP
                                               title  price_gbp  rating
0               Blood Defense (Samantha Brinkman #1)      20.30       3
1    

In [17]:
# pd.read_sql tho JOIN result
sql_result = pd.read_sql("""
    SELECT b.title, b.price_gbp, b.rating, c.category_name
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
    ORDER BY b.rating DESC, b.title ASC
    LIMIT 10
""", conn)

print("pd.read_sql result:")
print(sql_result)

# Same result pd.merge tho
df_categories = pd.read_sql("SELECT * FROM categories", conn)
df_books = pd.read_sql("SELECT * FROM books", conn)

merge_result = pd.merge(df_books, df_categories, on="category_id")
merge_result = merge_result[["title", "price_gbp", "rating", "category_name"]]
merge_result = merge_result.sort_values(
    ["rating", "title"], ascending=[False, True]
).head(10).reset_index(drop=True)

print("\npd.merge result:")
print(merge_result)

print("\nResults match:", sql_result["title"].tolist() == merge_result["title"].tolist())

pd.read_sql result:
                                               title  price_gbp  rating  \
0                 1,000 Places to See Before You Die      26.08       5   
1            A Flight of Arrows (The Pathfinders #2)      55.53       5   
2  A Spy's Devotion (The Regency Spies of London #1)      16.97       5   
3             A Time of Torment (Charlie Parker #14)      48.35       5   
4                                         Black Dust      34.53       5   
5                         Chase Me (Paris Nights #2)      25.27       5   
6                                       Mrs. Houdini      30.25       5   
7                              The Passion of Dolssa      28.32       5   
8                                       The Red Tent      35.66       5   
9                             Voyager (Outlander #3)      21.07       5   

        category_name  
0              travel  
1  historical-fiction  
2  historical-fiction  
3             mystery  
4             romance  
5         